# L03 -- Balanced Power Flow

Companion notebook for L03. We verify the lecture's central claim: **`pp.runpp` (balanced PF) cannot represent a single-phase load and hides per-phase violations** that `pp.runpp_3ph` reveals.

**Prerequisite**: L01, L02. Slides: `L03_balanced_pf.pdf`.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import pandapower as pp
from packaging.version import Version
import matplotlib.pyplot as plt

print(f"pandapower {pp.__version__}, numpy {np.__version__}, pandas {pd.__version__}")
assert Version(pp.__version__) >= Version("3.2.1"), "Need pandapower >= 3.2.1 (env: pdpower)"

## 1. A balanced LV feeder shared by two scenarios

2-bus feeder, $z = 0.05 + j0.05$ pu (positive-sequence) and $z_0 = 0.10 + j0.10$ pu (zero-sequence). 1 MVA / 0.4 kV base.

We will load it two ways:
- **Scenario A** (balanced): 0.4 pu total split as 0.133 pu per phase.
- **Scenario B** (unbalanced): 0.4 pu on phase A only.

In [ ]:
ZB = 0.16

def build_l03_feeder():
    net = pp.create_empty_network(sn_mva=1.0)
    b1 = pp.create_bus(net, vn_kv=0.4)
    b2 = pp.create_bus(net, vn_kv=0.4)
    pp.create_ext_grid(net, bus=b1, vm_pu=1.0,
        s_sc_max_mva=1e6, rx_max=0.1, x0x_max=1.0, r0x0_max=0.1)
    pp.create_line_from_parameters(net, b1, b2, length_km=1.0,
        r_ohm_per_km=0.05*ZB, x_ohm_per_km=0.05*ZB,
        r0_ohm_per_km=0.10*ZB, x0_ohm_per_km=0.10*ZB,
        c_nf_per_km=0.0, c0_nf_per_km=0.0, max_i_ka=10.0)
    return net, b2

# Scenario A: balanced 3-phase load. create_load is treated as 3-phase symmetric.
net_A, b2 = build_l03_feeder()
pp.create_load(net_A, bus=b2, p_mw=0.4, q_mvar=0.0)

## 2. Scenario A: both APIs agree

In [ ]:
pp.runpp(net_A)
print("Balanced PF (Scenario A):")
print(net_A.res_bus[["vm_pu"]].round(4))

pp.runpp_3ph(net_A)
print("\n3-phase PF (Scenario A):")
print(net_A.res_bus_3ph[["vm_a_pu", "vm_b_pu", "vm_c_pu", "unbalance_percent"]].round(4))

# All three phase magnitudes should match the balanced result
for phi in ["vm_a_pu", "vm_b_pu", "vm_c_pu"]:
    diff = abs(net_A.res_bus_3ph[phi][1] - net_A.res_bus.vm_pu[1])
    assert diff < 5e-4, f"Scenario A: {phi} disagrees with balanced PF"
print("Scenario A: balanced and 3-phase APIs agree (as expected).")

## 3. Scenario B: balanced PF lies

Replace the balanced load with a single-phase load on phase A. Same total power, very different per-phase result.

In [ ]:
net_B, b2 = build_l03_feeder()
pp.create_asymmetric_load(net_B, bus=b2,
    p_a_mw=0.4, q_a_mvar=0.0,
    p_b_mw=0.0, q_b_mvar=0.0,
    p_c_mw=0.0, q_c_mvar=0.0, type="wye")

# Balanced PF on the same network: it will see the asymmetric load as nonexistent unless we also add a balanced load.
# To compare apples-to-apples we ALSO put a 0.4 pu balanced load and pretend the operator only ran balanced PF.
net_B_bal, _ = build_l03_feeder()
pp.create_load(net_B_bal, bus=b2, p_mw=0.4, q_mvar=0.0)
pp.runpp(net_B_bal)

# 3-phase PF on the truly asymmetric setup
pp.runpp_3ph(net_B)

print("What balanced PF reports for 0.4 pu equivalent load:")
print(net_B_bal.res_bus[["vm_pu"]].round(4))
print("\nWhat 3-phase PF reports for 0.4 pu on phase A only:")
print(net_B_bal.res_bus_3ph_blank if False else net_B.res_bus_3ph[
    ["vm_a_pu", "vm_b_pu", "vm_c_pu", "unbalance_percent"]].round(4))

In [ ]:
# Quantify the gap: balanced says ~0.979, but phase A is actually ~0.908
v_balanced = net_B_bal.res_bus.vm_pu[1]
v_a = net_B.res_bus_3ph.vm_a_pu[1]
v_b = net_B.res_bus_3ph.vm_b_pu[1]
v_c = net_B.res_bus_3ph.vm_c_pu[1]
vuf = net_B.res_bus_3ph.unbalance_percent[1]

print(f"Balanced reports |V| = {v_balanced:.4f}   --> classified as SAFE")
print(f"3-phase truth:   |V_a| = {v_a:.4f}, |V_b| = {v_b:.4f}, |V_c| = {v_c:.4f}, VUF = {vuf:.2f}%")
print(f"Phase A violation: {v_a < 0.95}")
print(f"VUF violation:     {vuf > 2.0}")

## 4. Visualize the lie

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
phases = ["A", "B", "C"]
balanced_bars = [v_balanced]*3
three_phase = [v_a, v_b, v_c]
x = np.arange(3)
width = 0.35
ax.bar(x - width/2, balanced_bars, width, label="balanced PF", color="gray", alpha=0.7)
ax.bar(x + width/2, three_phase,   width, label="3-phase PF",  color=["#c82828", "#2828b4", "#148c28"])
ax.axhline(0.95, color="red", linestyle="--", alpha=0.5)
ax.axhline(1.05, color="red", linestyle="--", alpha=0.5)
ax.set_xticks(x); ax.set_xticklabels(phases)
ax.set_ylabel("$|V|$ (pu)"); ax.set_ylim(0.88, 1.06); ax.legend(); ax.grid(alpha=0.3)
ax.set_title("What balanced PF reports vs what 3-phase PF actually finds")
plt.tight_layout(); plt.show()

## Homework

### Required
1. Sweep the phase-A load from 0.05 to 0.65 pu. Plot $|V_a|$ vs load. Mark where it first crosses 0.95.
2. Replace the wye-connected asymmetric load with a delta connection (`type="delta"`). Explain the change.
3. Compute the VUF using the IEC definition ($|V^{(2)}|/|V^{(1)}|$) directly from your per-phase voltages and compare to `res_bus_3ph.unbalance_percent`.

### Optional
- Vary $Z_0/Z_1$ from 1 to 10 with phase-A load fixed at 0.4 pu. Plot $|V_a|$ vs $Z_0/Z_1$. Explain.

In [ ]:
# TODO Q1: phase-A load sweep
# ...

# TODO Q2: delta vs wye
# ...

# TODO Q3: VUF from Fortescue, compare to pandapower
# a = np.exp(2j*np.pi/3)
# ...